In [ ]:
import os
import requests
from Bio import SeqIO

# Folder to store downloaded PDB files
output_folder = "PDB files"
os.makedirs(output_folder, exist_ok=True)

# Path to your input FASTA file
fasta_path = "DbPtm/Test/dbptm_benchmark_filtered.csv"

# Track counts
downloaded = 0
not_found = 0

# Function to get UniProt ID from FASTA header
def extract_uniprot_id(header):
    try:
        parts = header.split('|')
        return parts[1] if len(parts) > 1 else None
    except Exception:
        return None

# Function to download PDB file from RCSB
def download_pdb(pdb_id, filename):
    url = f"https://alphafold.ebi.ac.uk/files/AF-{pdb_id}-F1-model_v4.pdb"
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, "w") as f:
            f.write(response.text)
        return True
    return False

# Process each entry in the FASTA file
for record in SeqIO.parse(fasta_path, "fasta"):
    header = record.description
    protein_id = header.split('|')[2].split(' ')[0]  # like 6PGL_MYCTU
    uniprot_id = extract_uniprot_id(header)

    if not uniprot_id:
        print(f"[SKIP] Could not extract UniProt ID for {protein_id}")
        not_found += 1
        continue

    save_path = os.path.join(output_folder, f"{protein_id}.pdb")
    success = download_pdb(uniprot_id, save_path)
    if success:
        downloaded += 1
        print(f"[{downloaded} downloaded, {not_found} not found] Saved {protein_id}.pdb (PDB: {uniprot_id})")
    else:
        not_found += 1
        print(f"[{downloaded} downloaded, {not_found} not found] Failed to download PDB: {uniprot_id}")

print("\nDone.")


[1 downloaded, 0 not found] Saved 40C1_ORYSJ.pdb (PDB: Q10M12)
[2 downloaded, 0 not found] Saved 6PGL_MYCTU.pdb (PDB: P9WQP5)
[2 downloaded, 1 not found] Failed to download PDB: A0A087WS16
[3 downloaded, 1 not found] Saved A0A0G2JUZ5_RAT.pdb (PDB: A0A0G2JUZ5)
[4 downloaded, 1 not found] Saved A0A0G2K4N7_RAT.pdb (PDB: A0A0G2K4N7)
[5 downloaded, 1 not found] Saved A0A0N7KI05_ORYSJ.pdb (PDB: A0A0N7KI05)
[6 downloaded, 1 not found] Saved A0A0P0VLH7_ORYSJ.pdb (PDB: A0A0P0VLH7)
[7 downloaded, 1 not found] Saved A0A0P0VPT6_ORYSJ.pdb (PDB: A0A0P0VPT6)
[8 downloaded, 1 not found] Saved A0A0P0VRI0_ORYSJ.pdb (PDB: A0A0P0VRI0)
[9 downloaded, 1 not found] Saved A0A0P0VTZ7_ORYSJ.pdb (PDB: A0A0P0VTZ7)
[10 downloaded, 1 not found] Saved A0A0P0VUB4_ORYSJ.pdb (PDB: A0A0P0VUB4)
[11 downloaded, 1 not found] Saved A0A0P0VW37_ORYSJ.pdb (PDB: A0A0P0VW37)
[12 downloaded, 1 not found] Saved A0A0P0WDR2_ORYSJ.pdb (PDB: A0A0P0WDR2)
[13 downloaded, 1 not found] Saved A0A0P0XBM6_ORYSJ.pdb (PDB: A0A0P0XBM6)
[14 down

In [6]:
import pandas as pd

test = pd.read_csv("DbPtm/dbptm_test_filtered.csv")
available_pdb = test['protein_id'].apply(lambda pid: os.path.isfile(os.path.join(output_folder, f"{pid}.pdb")))
num_available = available_pdb.sum()
print(f"PDB available for {num_available} out of {len(test)} entries.")

PDB available for 3897 out of 4210 entries.


In [2]:
import requests
from Bio import SeqIO

# Input FASTA file
fasta_path = "DbPtm/dbptm_test_filtered.fasta"

# Track counts
with_structure = 0
no_structure = 0

# Extract UniProt ID from FASTA header
def extract_uniprot_id(header):
    parts = header.split('|')
    return parts[1] if len(parts) > 1 else None

# Check RCSB API for experimental PDB structures
def has_experimental_pdb(uniprot_id):
    url = f"https://search.rcsb.org/rcsbsearch/v2/query?json="
    query = {
        "query": {
            "type": "terminal",
            "service": "text",
            "parameters": {
                "attribute": "rcsb_polymer_entity_container_identifiers.reference_sequence_identifiers.database_accession",
                "operator": "exact_match",
                "value": uniprot_id
            }
        },
        "return_type": "entry",
        "request_options": {
            "return_all_hits": True
        }
    }
    response = requests.post(url, json=query)
    if response.status_code == 200 and response.json().get("result_set"):
        return True
    return False

# Process each entry
for i, record in enumerate(SeqIO.parse(fasta_path, "fasta")):
    header = record.description
    uniprot_id = extract_uniprot_id(header)

    if not uniprot_id:
        print(f"[SKIP] Could not extract UniProt ID for record {i+1}")
        continue

    if has_experimental_pdb(uniprot_id):
        print(f"[FOUND] Experimental PDB structure for record {i+1}: {uniprot_id}")
        with_structure += 1
    else:
        print(f"[NOT FOUND] No experimental PDB structure for record {i+1}: {uniprot_id}")
        no_structure += 1

print(f"Proteins with experimental PDB structures: {with_structure}")
print(f"Proteins without experimental PDB structures: {no_structure}")


[NOT FOUND] No experimental PDB structure for record 1: Q10M12
[FOUND] Experimental PDB structure for record 2: P9WQP5
[NOT FOUND] No experimental PDB structure for record 3: A0A087WS16
[NOT FOUND] No experimental PDB structure for record 4: A0A0G2JUZ5
[NOT FOUND] No experimental PDB structure for record 5: A0A0G2K4N7
[NOT FOUND] No experimental PDB structure for record 6: A0A0N7KI05
[NOT FOUND] No experimental PDB structure for record 7: A0A0P0VLH7
[NOT FOUND] No experimental PDB structure for record 8: A0A0P0VPT6
[NOT FOUND] No experimental PDB structure for record 9: A0A0P0VRI0
[NOT FOUND] No experimental PDB structure for record 10: A0A0P0VTZ7
[NOT FOUND] No experimental PDB structure for record 11: A0A0P0VUB4
[NOT FOUND] No experimental PDB structure for record 12: A0A0P0VW37
[NOT FOUND] No experimental PDB structure for record 13: A0A0P0WDR2
[NOT FOUND] No experimental PDB structure for record 14: A0A0P0XBM6
[NOT FOUND] No experimental PDB structure for record 15: A0A0P0XU17
[NOT